## PseudoEARL-Net: a deep learning-based harmonization tool for retrospective EARL harmonization of multicenter FDG PET/CT

### Structure of the paper
1. Introduction
2. Materials and Methods
    2.1. Data Collection
    2.2. Data Preprocessing
    2.3. Model Architecture
    2.4. Training Procedure
    2.5. Evaluation Metrics
3. Results
    3.1. Quantitative Results
    3.2. Qualitative Results
4. Discussion
5. Conclusion

### Refs :
1. Revue de référence sur la philosophie et méthodologie EARL, impact sur SUV, MATV/TLG et utilisation pour EORTC/PERCIST et Deauville dans les études multicentriques
(https://link.springer.com/article/10.1007/s00259-017-3740-2) : Justifier le besoin de pseudo‑EARL
2. Travail sur la mise à jour des spécifications EARL pour les scanners TOF+PSF, montrant la faisabilité d’augmenter le contraste tout en restant harmonisable
(https://link.springer.com/article/10.1007/s00259-018-3977-4) : Lier à la robustesse des biomarqueurs
3. CNN pour reconnaître automatiquement les reconstructions EARL1/EARL2 vs non‑EARL, utile comme justification de l’importance de la conformité EARL dans les études IA
(https://link.springer.com/article/10.1186/s40658-022-00468-w) : Argumenter la place de l’IA dans la standardisation

4. SubtlePET (U‑Net 2.5D résiduel), très proche de votre approche : Validation clinique/phantom avec réduction de dose/temps (PET50, PET33), conservation SUV et qualité 
(https://doi.org/10.1186/s40658-022-00465-z)
5. Article “impact sur EORTC/PERCIST” : montre que comparer un PET standard à un PET débruité IA (SubtlePET, UHD vs EARL2) reste cliniquement acceptable pour les classifications EORTC/PERCIST, avec ~93 % de concordance et faible impact sur la prise en charge
(https://doi.org/10.1186/s13550-024-01128-z)

6. Revue sur les reconstructions DL (SubtlePET, AiCE, HYPER DLR, Precision DL) basée sur des architectures résiduelles/2.5D‑U‑Net
(https://doi.org/10.1007/s12149-025-02088-7)

### Introduction :
- Rôle du quantitatif FDG-PET et radiomics
SUV, MATV, TLG, radiomics pour pronostic/réponse, mais forte sensibilité aux protocoles et reconstructions

- Exposer le besoin d’harmonisation quantitative en PET multicentrique (SUV, radiomics, critères EORTC/PERCIST). Souligner les limites actuelles : absence d’EARL en routine dans certains centres, perte des données brutes, difficulté à inclure des cohortes historiques.
- Programme EARL, reconstructions EARL1/EARL2, impact démontré sur SUV et radiomics en multicentrique
(https://doi.org/10.2967/jnumed.119.229724,https://doi.org/10.1007/s00259-022-05919-1,https://doi.org/10.1007/s00259-017-3740-2,https://doi.org/10.1186/s40658-021-00390-7)
Limitation majeure : nécessite les données brutes ou une reconstruction EARL dédiée, difficile en rétrospectif
- Présenter l’état de l’art sur l’harmonisation (EARL1/EARL2), ses impacts cliniques et radiomiques
(https://doi.org/10.1007/s00259-017-3740-2,https://doi.org/10.1007/s00259-016-3441-2,https://doi.org/10.1007/s00259-018-4151-8,https://doi.org/10.1186/s40658-017-0185-4)

- Introduire les approches classiques (filtrage gaussien) et leurs limites 
(https://doi.org/10.1186/s40658-019-0257-8,https://doi.org/10.1016/j.ejmp.2017.10.052)
- Harmonisation post‑reconstruction : ComBat et autres approches pour harmoniser les features (radiomics, SUV) sans modifier les images.
Limites : pas d’accès à une image harmonisée, difficile pour la relecture clinique, segmentation, ou pour introduire de nouveaux descripteurs.

- Positionner la contribution : génération d’images pseudo-EARL à partir de PET standard via deep learning résiduel (UNet), pour permettre l’inclusion rétrospective de données non conformes
- Aucun travail ne propose explicitement une traduction standard → EARL supervisée, optimisée pour radiomic reproducibility avec très peu de sujets par centre
- Pas de méthode publiée générant explicitement des images pseudo‑EARL à partir de PET standard haute résolution (UHD, PSF, etc.), alors que de nombreux réseaux résiduels U‑Net sont déjà utilisés pour le débruitage/harmonisation implicite

Il y a aussi :
- Rôle des critères EORTC/PERCIST pour le monitoring thérapeutique
- Montée en puissance de la reconstruction avancée (TOF/PSF, BPL) et du DL (SubtlePET, autres U‑Net résiduels) et risque de perturber la comparabilité quantitative
- 


### Données :
- Décrire les 4 cohortes (3 centres, EARL1/EARL2) et la conformité aux guidelines EANM/EARL
(https://doi.org/10.1007/s00259-022-05919-1, https://doi.org/10.1007/s00259-017-3740-2)
- Description précise des cohortes (Rouen1/2, Rennes, Nantes), nombre de sujets, types de scanners, protocoles d’acquisition/reconstruction.
Préciser EARL1/EARL2 selon les cohortes
- Préciser que les mêmes données brutes ont été reconstruites en standard et EARL GT, comme dans les études EARL/radiomics
(https://doi.org/10.2967/jnumed.119.229724, https://doi.org/10.1186/s40658-021-00390-7)
- Décrire précisément les paramètres standard vs EARL1/EARL2 (TOF, PSF, FWHM), en s’alignant sur les recommandations de Boellaard/Aide/van Sluis
(https://doi.org/10.2967/jnumed.119.229724, https://doi.org/10.1007/s00259-022-05919-1, https://doi.org/10.1007/s00259-017-3740-2, https://doi.org/10.21037/qims-22-443)


### Modèle :
- Justification par rapport à l'état de l'art
- U‑Net résiduel 2D‑to‑3D, inspiré des architectures ResUNet/R2U‑Net/MAGRes‑UNet, connues pour mieux exploiter les détails avec peu de données
(https://doi.org/10.1016/j.bspc.2021.102643, https://doi.org/10.1109/access.2024.3374108, https://doi.org/10.1117/1.jmi.6.1.014006)
- Stratégie Δ‑map
- Normalisation log‑SUV et patches 2D→3D : Lier la normalisation log à la capacité des modèles DL à couvrir à la fois fond faiblement captant et lésions très avides
(https://doi.org/10.1055/a-2198-0358, https://doi.org/10.1007/s12194-024-00780-3, https://doi.org/10.1007/s00259-022-05746-4, https://doi.org/10.1016/j.media.2023.103046)
- Stratégie data‑efficient

### Evaluation :
1. Evaluation voxel wise
- SNR/SNR foie, COV, PSNR/SSIM entre pseudo‑EARL et GT EARL
- Cartes d’erreur pour illustrer la localisation des écarts.

2. Évaluation SUV (quantification classique)
- aRE (%) pour SUVmean/SUVmax/SUVpeak au foie, lésion, et éventuellement autres organes de référence
(https://doi.org/10.2967/jnumed.119.229724,https://doi.org/10.1007/s00259-022-05919-1,https://doi.org/10.1007/s00259-017-3740-2,https://doi.org/10.1186/s40658-021-00390-7)
- Bland–Altman et/ou ICC pour SUV, en les comparant aux marges de variabilité observées dans EARL et études multicentriques
- SUV/SUL/SUVpeak/SUVmax, MATV/TLG, radiomics sur lésions et organes de référence EARL vs pseudo‑EARL.
Si possible, re‑calcul des classifications EORTC/PERCIST sur standard vs pseudo‑EARL (parallèle à Weyts 2024 : standard vs AI‑denoised)

3. Évaluation radiomics
- Justifier le choix des 93 features par rapport aux recommandations IBSI et travaux sur robustesse via phantoms/EARL
- CCC et Bland–Altman (biais, limites d’accord) pseudo‑EARL vs EARL GT
- Variabilité inter‑centre non harmonisée (si possible via un baseline “standard vs standard” entre centres) ??? (harmonisation)

4. Analyse secondaire : classification / downstream task
- Définir un mini‑tâche prédictive (par ex. simple classification binaire sur un endpoint disponible ou au minimum tissue classification foie/splénomédu osseux)
- prédiction métastases à distance ?
- Montrer sur la tâche que le pseudo‑EARL se rapproche d’EARL GT et surpasse le standard

5. Analyse de sensibilité par centre / par type de feature
6. Analyse de l'impact selon les critères EORTC/PERCIST (SUVmax vs SULpeak)
7. Analyse de l'impact de la taille de l’échantillon d’entraînement (learning curve)


### Potentielles discussions :
1. RECIST/PERCIST/EORTC :  
Tu n'as pas besoin de recalculer les critères PERCIST ou EORTC sur tes 363 patients pour ce papier (tes métriques de SUV et de radiomique suffisent amplement). En revanche, tu dois utiliser ces concepts dans ta Discussion pour donner de la hauteur clinique à ton travail.Voici l'argumentaire en 3 points que tu peux rédiger :L'ancrage clinique : Rappeler que la quantification TEP ($SUV_{max}$ / $SUL_{peak}$) régit les critères EORTC et PERCIST pour évaluer la réponse thérapeutique dans les cancers solides (comme le NSCLC).Le problème des données historiques : Expliquer que dans les essais cliniques multicentriques rétrospectifs, l'absence de standardisation EARL biaise ces classifications, car la variabilité inter-scanner est confondue avec la vraie réponse biologique au traitement.L'ouverture (La référence à SubtlePET et Weyts et al.) : C'est le point soulevé par l'autre IA et il est excellent. Des logiciels commerciaux de Deep Learning (comme SubtlePET, qui fait du débruitage) ont été critiqués dans la littérature (notamment par Weyts et al.) parce qu'en modifiant l'image, ils modifient artificiellement le SUV et donc les scores EORTC/PERCIST.Ta force : Contrairement à un simple débruitage agressif, ton modèle PseudoEARL-Net prédit spécifiquement le filtre résiduel ($\Delta$-map) physique nécessaire pour atteindre la cible EARL. Tes résultats de reproductibilité radiomique (CCC $> 0.93$) prouvent que ton IA préserve la texture et la linéarité du signal, ce qui garantit une classification EORTC/PERCIST hautement sécurisée pour le clinicien.

2. (A voir) Harmonisation post‑reconstruction côté “features”:  
ComBat et autres approches pour harmoniser les features (radiomics, SUV) sans modifier les images 
(https://doi.org/10.2967/jnumed.117.199935, https://doi.org/10.2967/jnumed.121.263102).
Limites : pas d’accès à une image harmonisée, difficile pour la relecture clinique, segmentation, ou pour introduire de nouveaux descripteurs.


## Method
1. Problem Formulation and Residual Learning  

Harmonizing standard PET images to meet EARL reconstruction standards presents a unique challenge in deep learning: the source (Standard PET) and target (EARL PET) images are structurally and visually highly similar. A conventional image-to-image translation approach ($F(X) \rightarrow Y$) inevitably converges toward a local minimum of identity mapping, where the network learns to simply reproduce the input image without applying the required subtle filtering ($F(X) \approx X$).

To circumvent this issue, we reformulated the task as a residual learning problem. Rather than synthesizing the entire EARL image from scratch, the model is optimized to isolate and predict the exact difference map (the residual) between the standard and EARL reconstructions. This approach is mathematically expressed as:
$$\hat{R} ​≈ α(Y − X​)$$

Where $\hat{R}$ represents the predicted residual, $Y$ is the target EARL image, $X$ is the input standard PET image and $\alpha$ is a scalar amplification factor. By focusing on learning the residual, the model is encouraged to capture only the necessary adjustments needed to transform the standard PET into its EARL counterpart, effectively bypassing the identity mapping pitfall and enhancing the learning of subtle features that distinguish the two image types.

The finalized EARL image $\hat{Y}$ is then reconstructed by de-amplifying this prediction before adding it back to the original input image:$$\hat{Y} = X + \frac{\hat{R}_{amp}}{\alpha}$$

This design (i) forces the network to focus on the small, clinically relevant differences between acquisition/reconstruction standards and (ii) preserves the original macroscopic PET signal because the final EARL image is reconstructed by adding a (de-amplified) residual to the input image rather than synthesizing the full image from scratch.

2. U-Net based 2D-to-3D translation approach  

The transformation from a standard PET space to the corresponding EARL image is fundamentally an operation of local filtering and Point Spread Function modification. By nature, this process is anatomy-agnostic and depends on local voxel interactions rather than macroscopic anatomical structures. We exploit this property afin de maximiser l'apport d'un ensemble d'entrainement limité., we opted for a patch-based UNet translation model using $64 \times 64$ crops, 

-  By decomposing volumes into individual slices, we can create larger twodimensional datasets with increased modes variability and diversity
- deconstruction of the global three-dimensional dataset into smaller, localized patches permet au model to solely focus on the translation task, allow une convergence with only few training samples
- which allows the network to learn local filtering patterns effectively  
- maintaining computational efficiency. 

The network takes as input a localized patch of the standard PET image with a minimal spatial context of neighbouring slices (5 in our case), and outputs the corresponding patch of the EARL image.

The global training objective is a weighted sum of two complementary terms that operate in the absolute SUV spaces and SSIM term as an additional constraint to ensure the preservation of the luminance, contrast, and structure of the original image:
$$\mathcal{L} = \lambda_{SUV} \cdot \mathcal 
{L}_{SUV} + \lambda_{SSIM} \cdot \mathcal{L}_{SSIM}$$



3. Non-Linear Data Normalization  

Standardized Uptake Values (SUV) in PET imaging exhibit a highly skewed distribution, dominated by low-intensity background noise with sparse but intense physiological or tumoral "hot-spots" (e.g., brain, bladder, lesions). To stabilize the learning of the residual, we apply (percentile normalization) followed by a logarithmic transformation to the SUV values, which compresses the dynamic range and mitigates the influence of extreme values. The patches are scaled into [-1, 1] given the maximum log suv value in the training set, which is determined by the 99.9th percentile of the SUV distribution to avoid outliers dominating the scaling. This non-linear normalization strategy ensures that the network can effectively learn from both low and high-intensity regions without being biased towards the extreme values, thus enhancing the overall performance of the model in harmonizing PET images to EARL standards.